# Reference: SIM Portfolio Problems
This page collects the two portfolio problems that the week 7 examples solve with single index model (SIM) inputs, and the one result they rely on, stated as in the L7a lecture. [The tangent portfolio notebook](CHEME-5660-L7b-Algorithm-TangentContinuation-Fall-2026.ipynb) derives the result.

___


## Notation
The single index model of L6b writes the growth rate of risky asset $i$ in the chosen set $\mathcal{P}$ as $g_{i} = \alpha_{i} + \beta_{i}\,g_{M} + \varepsilon_{i}$, where $g_{M}$ is the market growth rate and $\varepsilon_{i}$ is a residual with zero mean, uncorrelated with the market and across assets. Estimating the model on training data and collecting the intercepts and betas in $\hat{\boldsymbol{\alpha}}$ and $\hat{\boldsymbol{\beta}}$ gives the portfolio inputs:

$$
\boxed{\begin{aligned}
\hat{\mathbf{g}}_{\mathrm{SIM}} &= \hat{\boldsymbol{\alpha}}+\hat{\boldsymbol{\beta}}\,g'_M &&\quad\text{mean growth rates}\\
\hat{\mathbf{\Sigma}}_{g,\mathrm{SIM}} &= \underbrace{s^2_{g,M}\hat{\boldsymbol{\beta}}\hat{\boldsymbol{\beta}}^\top}_{\text{market}}+\underbrace{\hat{\mathbf{D}}_g}_{\text{residual}} &&\quad\text{growth-rate covariance}.
\end{aligned}}
$$

Here, $g'_M$ and $s^2_{g,M}$ are the sample mean and variance of the market growth rate, and the diagonal matrix $\hat{\mathbf{D}}_g$ holds the estimated residual variances $s^2_{g,\varepsilon,i}$. Growth rates and intercepts have units of inverse years, betas are dimensionless, and covariance entries have units of inverse years squared. The symbols used on this page are:

* $\mathbf{w}\in\mathbb{R}^{|\mathcal{P}|}$: the weights, with $w_{i}$ the fraction of initial wealth in risky asset $i$, and $\mathbf{1}$ the vector of ones.
* $g_{\star}$: the growth target, in inverse years, and $g_{f}$: the risk-free growth rate, in inverse years.
* $w_{f} = 1 - \mathbf{1}^{\top}\mathbf{w}$: the fraction of wealth in the risk-free asset, which is negative when we borrow, and $\theta = 1 - w_{f}$: the risky fraction.
* $\mathbf{w}_{\mathcal{T}}$: the weights of the tangent portfolio $\mathcal{T}$, which sum to one.

___


## SIM-1: Long-only minimum-variance portfolio
The risky-asset-only problem. The weights sum to one, so there is no risk-free holding, and a growth floor selects a point on the long-only efficient frontier.

> __Long-only minimum-variance portfolio with SIM inputs:__
>
> __Setup.__ Choose the weights $\mathbf{w}\in\mathbb{R}^{|\mathcal{P}|}$, where $w_{i}$ is the initial fraction invested in asset $i$. Given the SIM inputs and a floor $g_{\star}$ on estimated expected growth, in inverse years, we solve:
>
> $$\boxed{\begin{aligned} \underset{\mathbf{w}}{\operatorname{minimize}}\quad &\underbrace{s^{2}_{g,M}\left(\hat{\boldsymbol{\beta}}^{\top}\mathbf{w}\right)^{2}}_{\text{market risk}}+\underbrace{\sum_{i\in\mathcal{P}}s^{2}_{g,\varepsilon,i}\,w_{i}^{2}}_{\text{residual risk}} &&\quad\text{minimize }\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g,\text{SIM}}\mathbf{w}\\ \text{subject to}\quad &\hat{\mathbf{g}}_{\text{SIM}}^{\top}\mathbf{w}\geq g_{\star} &&\quad\text{meet or exceed the growth target}\\ &\mathbf{1}^{\top}\mathbf{w}=1 &&\quad\text{weights sum to one}\\ &0\leq w_{i}\leq1,\quad i\in\mathcal{P} &&\quad\text{exclude short positions}. \end{aligned}}\tag{SIM-1}$$
>
> __Solution.__ A convex quadratic program, solved numerically. Sweeping $g_{\star}$ upward from the growth rate of the long-only global minimum-variance (GMV) portfolio traces the efficient frontier. Setting $g_{\star}$ at or below the smallest entry of $\hat{\mathbf{g}}_{\mathrm{SIM}}$ makes the floor inactive and returns the GMV portfolio.

___


## SIM-2: Long-only risky holdings with risk-free lending or borrowing
The risky and risk-free problem. Once we can lend or borrow, the risky weights no longer need to sum to one, so we keep track of two sets of weights. The __fund weights__ $\mathbf{w}_{\mathcal{T}}$ give the mix within the risky fund, and the __wealth weights__ $\mathbf{w}$ give each risky holding as a fraction of initial wealth. They differ by the risky fraction $\theta$:

$$
\mathbf{w}=\theta\,\mathbf{w}_{\mathcal{T}},\qquad
\mathbf{1}^{\top}\mathbf{w}_{\mathcal{T}}=1,\qquad
\theta=\mathbf{1}^{\top}\mathbf{w}=1-w_{f}.
$$

> __Long-only risky holdings with risk-free lending or borrowing:__
>
> __Setup.__ Assume the SIM covariance is positive definite and at least one estimated mean growth rate exceeds $g_f$. For a feasible target $g_{\star}>g_f$, choose $\mathbf{w}$ to solve:
>
> $$\boxed{\begin{aligned} \underset{\mathbf{w}}{\operatorname{minimize}}\quad &\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g,\mathrm{SIM}}\mathbf{w} &&\quad\text{minimize growth-rate variance}\\ \text{subject to}\quad &(\hat{\mathbf{g}}_{\mathrm{SIM}}-g_f\mathbf{1})^{\top}\mathbf{w}\geq g_{\star}-g_f &&\quad\text{meet the excess-growth target}\\ &0\leq w_i\leq1,\quad i\in\mathcal{P} &&\quad\text{long-only holdings with per-asset caps}. \end{aligned}}\tag{SIM-2}$$
>
> __What are the constraints?__ We do **not** impose $\mathbf{1}^{\top}\mathbf{w}=1$ in SIM-2. The budget is $\mathbf{1}^{\top}\mathbf{w}+w_f=1$, and $w_f$ may be negative. The lower bound $w_i\geq0$ prohibits shorting risky assets. The upper bound $w_i\leq1$ is an additional limit: no single risky holding may exceed 100% of initial wealth. Unlike the risky-only problem, this upper bound does not follow automatically from the budget.
>
> __How do we solve it?__ SIM-2 is a convex quadratic program. We solve it numerically, allowing the solver to determine which risky holdings are zero.

___


## SIM-3: One risky direction
The result the examples rely on. As long as no holding hits its 100% limit, $w_{i}\leq1$, the target in SIM-2 sets only how much of the risky fund we hold, not the mix inside it:

> __One risky direction:__
>
> $$\boxed{\mathbf{w}(g_{\star})=(1-w_{f})\,\mathbf{w}_{\mathcal{T}},\qquad 1-w_{f}=\frac{g_{\star}-g_{f}}{\hat{\mathbf{g}}_{\mathrm{SIM}}^{\top}\mathbf{w}_{\mathcal{T}}-g_{f}}}\tag{SIM-3}$$
>
> where the long-only tangent portfolio $\mathbf{w}_{\mathcal{T}}=\mathbf{w}/\mathbf{1}^{\top}\mathbf{w}$ is any solution rescaled to sum to one. Targets below $\hat{\mathbf{g}}_{\mathrm{SIM}}^{\top}\mathbf{w}_{\mathcal{T}}$ lend, and targets above it borrow.

So one solve of SIM-2 at any target where no holding hits the 100% limit, rescaled to sum to one, gives $\mathbf{w}_{\mathcal{T}}$, the long-only fund with the largest Sharpe ratio, and $\hat{\mathbf{g}}_{\mathrm{SIM}}^{\top}\mathbf{w}_{\mathcal{T}}$ is its expected growth rate. [The tangent portfolio notebook](CHEME-5660-L7b-Algorithm-TangentContinuation-Fall-2026.ipynb) derives this result and compares the single solve with the search from the L7a lecture.

___


## Where each problem is used
| Problem | Solved in | How |
|---|---|---|
| SIM-1 | the L7a risky-asset-only example; the GMV portfolio in [the utility allocator example](CHEME-5660-L7b-Example-Utility-Allocator-Fall-2026.ipynb) | `MyMarkowitzRiskyAssetOnlyPortfolioChoiceProblem` |
| SIM-2 | the L7a risky and risk-free example; the tangent sweep in [the capital allocation line example](CHEME-5660-L7b-Example-CAL-Optimal-Allocation-Fall-2026.ipynb); the single solve and the continuation check in the utility allocator example | `MyMarkowitzRiskyRiskFreePortfolioChoiceProblem` |
| SIM-3 | rescaling the SIM-2 solutions in both L7b examples, derived in [the tangent portfolio notebook](CHEME-5660-L7b-Algorithm-TangentContinuation-Fall-2026.ipynb) | divide the wealth weights by their sum |

The problem types are documented on the [course package's portfolio page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/portfolio/).

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___
